In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W15 出發！本週目標：自己寫去識別化程式，並自己寫資料平衡檢查")
print("同一把尺量兩次：先量別人的 App 與擴充功能，再量自己的專題資料")
print("本週要自己補四個空：雜湊兩行、總筆數、各類比例")

# W15　權限稽核、去識別化與資料平衡（電腦教室版）

**先做稽核（不吃網路），再回來寫程式。**

**任務一：兩邊各查五個**
- 手機 App：設定 → 隱私權／權限管理員，挑你每天用最久的五個。
- Chrome 擴充功能：網址列輸入 `chrome://extensions`，挑五個（不到五個就全查）。
- 一項一項抄下來：名稱、要了哪些權限、合不合理、關掉會怎樣。
- 只要你**講不出它為什麼要**，就在後面打一個問號。

**判斷的標準只有一句話**：不給這個權限，它的主要功能還能用嗎？
地圖要定位很合理，手電筒要定位就不合理。

**開始之前**：功能表「檔案 → 在雲端硬碟中儲存副本」。

**教室電腦一個擴充功能都沒有**（每次開機會還原）：改查自己的筆電或手機瀏覽器，
同組互相看也可以。

### 第 1 格：把稽核結果算一算

**這一格要做什麼**：把 `rows` 換成你自己查到的**十筆**（五個 App＋五個擴充功能）。
沒有空格，但資料一定要換成你自己的。

**寫對了會看到什麼**：一張分成 `App` 與 `擴充` 兩列的統計表，
以及「說不出理由的比例」。超過 0.3 會多印一行提醒你回去逐項關掉或降級。

**看哪一邊比較誇張**：`groupby` 會把 App 與擴充功能分開算，這就是本週的討論素材。

In [ ]:
#@title 第 1 格：稽核統計
import pandas as pd
rows = [
    {"對象": "IG", "類型": "App", "權限數": 6, "問號": 2},
    {"對象": "手電筒", "類型": "App", "權限數": 4, "問號": 3},
    {"對象": "翻譯外掛", "類型": "擴充", "權限數": 3, "問號": 2},
]
df = pd.DataFrame(rows)
print(df.groupby("類型")[["權限數", "問號"]].sum())
w = df["問號"].sum() / df["權限數"].sum()
print("說不出理由的比例 =", round(w, 2))
if w > 0.3:
    print("超過三成講不出理由，回去逐項關掉或降級")

### 第 2 格：自己寫去識別化

去識別化不是把姓名刪掉就好，要讓**同一個人每次都變成同一個代號**，
這樣資料才串得起來，但別人看不出是誰。

**這一格要做什麼**：補兩行。
- 第一行：把 `salt` 和 `x` 接起來再 `.encode()`。
  字串要先轉成位元組，`sha256` 只吃 bytes 不吃字串。
- 第二行：`sha256` 之後取 `hexdigest()` 的**前 8 碼**（原本是 64 個字，太長）。

**`salt` 是什麼、為什麼一定要加**：沒加 salt，
別人把全班學號都雜湊一次就能反查出是誰。`salt` 一組只能有一個，寫在最上面。

**寫對了會看到什麼**：印出三個代號，最後一行印出 `True`——
代表同一個學號永遠得到同一個代號。

**交出去之前**：姓名與學號那兩欄要**整欄刪掉**，簡報與 Demo 也不能出現。

In [ ]:
#@title 第 2 格：自己寫 hide()
import hashlib
df = pd.DataFrame({"學號": ["4B0A001", "4B0A002", "4B0A001"],
                   "姓名": ["王小明", "林小美", "王小明"]})
salt = "groupA"
def hide(x):
    s = ____           # ← 自己寫：salt 和 x 接起來再 .encode()
    return ____        # ← 自己寫：sha256 後取前 8 碼
df["代號"] = df["學號"].apply(hide)
print(df[["代號"]])
print("兩筆同學號代號一樣嗎：", df["代號"][0] == df["代號"][2])

### 第 3 格：自己寫資料平衡檢查

各類筆數差太多，模型會學會一直猜多數那一類，正確率看起來很高卻沒有用。

**這一格要做什麼**：補兩行。
- 第一行：全部加起來（`Series` 有現成的方法）。
- 第二行：每一類佔的比例＝各類筆數 ÷ 總筆數。

**寫對了會看到什麼**：一張兩欄的表（`筆數`、`比例`），
比例那一欄加起來要等於 1；再印出最多與最少差幾倍。
差超過三倍會多印一行警告。

**把 `plan` 換成你們組實際打算蒐集的數量**，這就是第 16 週要收多少資料的依據。

In [ ]:
#@title 第 3 格：自己寫平衡檢查
plan = {"紙類": 120, "寶特瓶": 100, "鋁罐": 30}
s = pd.Series(plan)
total = ____          # ← 自己寫：全部加起來
ratio = ____          # ← 自己寫：每一類佔的比例
print(pd.DataFrame({"筆數": s, "比例": ratio.round(3)}))
r = s.max() / s.min()
print("最多與最少差", round(r, 1), "倍")
if r > 3:
    print("差超過三倍，模型會學會一直猜多數那一類")

### 第 4 格（進階）：把它包成一個 `check_bias()`

把前面那段包成一個函式，之後每次收完資料都跑一次。沒有空格，直接執行。

**寫對了會看到什麼**：一份五行的檢查報告——類別數、總筆數、最少的一類、
最多／最少幾倍，最後一行是「需要補資料」或「可以開始訓練」。

**什麼時候要跑它**：第 16 週收完資料、第 17 週訓練之前，各跑一次。

In [ ]:
#@title 第 4 格（進階）：check_bias()
def check_bias(counts, name="資料"):
    s = pd.Series(counts)
    r = s.max() / s.min()
    print("====", name, "檢查報告 ====")
    print("類別數 =", len(s), "總筆數 =", s.sum())
    print("最少的一類：", s.idxmin(), s.min(), "筆")
    print("最多／最少 =", round(r, 1), "倍")
    print("結論：", "需要補資料" if r > 3 else "可以開始訓練")

check_bias({"紙類": 120, "寶特瓶": 100, "鋁罐": 30}, "垃圾分類")

### 任務五：分組與專題題目登記（今天下課前一定要交）

1. 先把組別喬定（3 到 4 人，不接受個人單獨進行）。
2. 每人講一個想法，一人 30 秒，先不要評論，把想法全部倒出來。
3. 挑一個並縮小：用**一句話**寫出輸入是什麼、輸出是什麼。
4. 想清楚資料怎麼收：用什麼收？要幾筆？誰負責收？誰負責標？
5. 填登記表：組員、題目、資料來源。

**做得完的六個方向**：校園垃圾分類、手勢控制、通勤方式辨識、口音辨識、
校園噪音地圖、課程評語分析。

**專題規格重點**：資料每類至少 100 筆；程式要有自己寫的部分（口試會請你解釋任何一行）；
第 18 週要有可以現場操作的 Demo。

### 收工：延伸挑戰與繳交

- **A**（每個人都要做完）：把 `plan` 換成你們組實際打算蒐集的數量，
  用 `check_bias()` 印出一份檢查報告。
- **B**：把學號換成你們四個人的，確認四個代號都不一樣，也看不出誰是誰。
- **C**：說出「拿掉性別欄位就不會有性別偏見」為什麼是錯的，
  舉一個會間接洩漏的欄位。

**常見狀況**：`hashlib` 說要 bytes＝忘了 `.encode()`；
兩筆同學號代號不同＝`salt` 中途被改掉了；
截圖拍到帳號姓名＝上傳前先用小畫家塗掉。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：手機 App 與擴充功能的稽核表（各五個）、第 1 格的統計與比例")
print("以及寫完的 .ipynb（要含 hide() 與平衡檢查）")
print("檔名：AI導論_W15_學號_姓名，專題登記表今天下課前完成")
print("提醒：交出去的資料只留代號欄，姓名與學號整欄刪掉")

---

<details>
<summary>參考解（四個空格都自己試過再打開）</summary>

```python
# 第 2 格
def hide(x):
    s = (salt + x).encode()
    return hashlib.sha256(s).hexdigest()[:8]

# 第 3 格
total = s.sum()
ratio = s / total
```

為什麼是這樣寫：

- `sha256` 只吃位元組，所以字串要先 `.encode()`。
  忘了寫會看到 `TypeError: Strings must be encoded before hashing`。
- `hexdigest()` 回傳 64 個十六進位字元，取前 8 碼只是為了**方便閱讀**，
  不是為了安全——這一點不要搞混。
- salt 加在前面或後面都可以，但**全組必須一致**，
  中途改掉，同一個人就會變成兩個代號。
- `s / total` 是整個 Series 除以一個數字，pandas 會自動每一項都除，
  不必寫迴圈。

</details>